
# Exp.04C — Fixed-Checkpoint Refine-Conditioning Intervention

This experiment uses the **completed Exp.04B best-FID Refine Transformer checkpoint** and
does **not retrain any model**.

Five fixed-checkpoint conditions are evaluated:

| Condition | Intervention |
|---|---|
| `full` | Original Refine Transformer |
| `no_text` | Zero only the Refine-stage text/keyword conditioning embedding. The Base Transformer still receives text. |
| `no_pose` | Zero the Refine-stage pose-code embedding. |
| `no_stage` | Zero only the explicit RVQ-stage embedding. Stage-specific output-head selection and residual-history indexing are retained. |
| `no_history` | Zero the cumulative preceding residual-stage embedding. |

These are **causal sensitivity interventions on one trained checkpoint**, not matched retraining
ablations.

Evaluation uses the HumanML3D test split, public `eval_refine_trans.py`, fast vectorized
inference, 20 repeats per condition, and the same seed reset for every condition so that
condition differences are paired under a common random-number sequence.


In [ ]:

from pathlib import Path
import os, sys, json, re, time, hashlib, subprocess, shlex

PGR = Path(".")
PY = Path("venv/bin/python")

PRE = PGR / "revision_preflight"
RESULTS = PGR / "revision_results" / "exp04c_conditioning"
LOGDIR = PGR / "revision_logs" / "exp04c_conditioning"
EVAL_OUT = RESULTS / "eval_runs"

B_AUDIT = PGR / "revision_results" / "exp04b_refine" / "exp04b_refine_completion_audit.json"
A_AUDIT = PGR / "revision_results" / "exp04a_base" / "exp04a_base_completion_audit.json"

WRAPPER = PRE / "eval_refine_trans_exp04c_intervention.py"
QUEUE = PRE / "run_exp04c_conditioning_queue.py"

PIDFILE = LOGDIR / "exp04c_queue.pid"
QUEUE_LOG = LOGDIR / "exp04c_queue.log"
LAUNCH_JSON = LOGDIR / "exp04c_launch.json"

for d in [PRE, RESULTS, LOGDIR, EVAL_OUT]:
    d.mkdir(parents=True, exist_ok=True)

assert PY.exists(), PY
assert PGR.exists(), PGR
assert B_AUDIT.exists(), B_AUDIT
assert A_AUDIT.exists(), A_AUDIT

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

b = json.loads(B_AUDIT.read_text())
a = json.loads(A_AUDIT.read_text())

REFINE_DIR = Path(b["run_dir"])
REFINE_CKPT = Path(b["best_fid_checkpoint"])
BASE_DIR = Path(b["base_dir"])
BASE_CKPT = BASE_DIR / "net_best_fid.pth"
TOKENIZER_DIR = Path(b["tokenizer_dir"])
TOKENIZER_CKPT = TOKENIZER_DIR / "net_best_fid.pth"

for p in [
    REFINE_DIR / "arguments.yaml", REFINE_CKPT,
    BASE_DIR / "arguments.yaml", BASE_CKPT,
    TOKENIZER_DIR / "arguments.yaml", TOKENIZER_CKPT,
    PGR / "eval_refine_trans.py",
    PGR / "models" / "rt2m_trans.py",
]:
    assert p.exists(), p

print("=== EXP.04C FROZEN STACK ===")
print("Python       :", sys.executable)
print("Tokenizer    :", TOKENIZER_CKPT)
print("Base         :", BASE_CKPT)
print("Refine       :", REFINE_CKPT)
print("Refine iter  :", b["checkpoint_nb_iter"])
print("Refine SHA256:", sha256(REFINE_CKPT))

assert int(b["checkpoint_nb_iter"]) == 10000
print("[OK] Exp.04B best-FID Refine checkpoint resolved.")


In [ ]:
# 2. Build the intervention/evaluation wrapper.
wrapper_text = '\nimport os, sys\nfrom pathlib import Path\n\nPGR = Path(".")\nCLIP_ROOT = PGR / "revision_preflight" / "third_party" / "openai_clip_dcba3cb2e282"\n\nos.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0")\nos.chdir(PGR)\nsys.path.insert(0, str(PGR))\nsys.path.insert(0, str(CLIP_ROOT))\n\nimport numpy as np\nif not hasattr(np, "float"):\n    np.float = float\nif not hasattr(np, "int"):\n    np.int = int\nif not hasattr(np, "bool"):\n    np.bool = bool\n\nimport dataset.dataset_TM_eval as _tm_eval\nif not hasattr(_tm_eval, "get_subsamples_loader"):\n    def _unused_subsamples(*args, **kwargs):\n        raise NotImplementedError("Exp.04C disables qualitative drawing.")\n    _tm_eval.get_subsamples_loader = _unused_subsamples\n    print("[OK] Added dataset_TM_eval.get_subsamples_loader import shim")\n\nimport torch\nimport models.rt2m_trans as _rt\n\ncondition = os.environ.get("EXP04C_CONDITION", "full").strip().lower()\nvalid = {"full", "no_text", "no_pose", "no_stage", "no_history"}\nif condition not in valid:\n    raise ValueError(f"Unknown EXP04C_CONDITION={condition!r}; expected {sorted(valid)}")\n\n_orig_cond = _rt.InputProcessor.emb_cond\n_orig_pose = _rt.InputProcessor.emb_pose_codes\n_orig_stage = _rt.InputProcessor.emb_q_ids\n_orig_hist = _rt.InputProcessor.emb_res_codes\n\nif condition == "no_text":\n    def _zero_cond(self, clip_feature):\n        return torch.zeros_like(_orig_cond(self, clip_feature))\n    _rt.InputProcessor.emb_cond = _zero_cond\n\nelif condition == "no_pose":\n    def _zero_pose(self, p_codes):\n        return torch.zeros_like(_orig_pose(self, p_codes))\n    _rt.InputProcessor.emb_pose_codes = _zero_pose\n\nelif condition == "no_stage":\n    def _zero_stage(self, q_ids):\n        return torch.zeros_like(_orig_stage(self, q_ids))\n    _rt.InputProcessor.emb_q_ids = _zero_stage\n\nelif condition == "no_history":\n    def _zero_history(self, all_indices, active_q_layers, inference=False):\n        return torch.zeros_like(\n            _orig_hist(self, all_indices, active_q_layers, inference=inference)\n        )\n    _rt.InputProcessor.emb_res_codes = _zero_history\n\nprint(f"[OK] Exp.04C intervention installed: {condition}")\n\nif os.environ.get("EXP04C_DRYRUN", "0") == "1":\n    proc = _rt.InputProcessor(\n        num_vq=392, num_rvq=512, clip_dim=512, embed_dim=1024,\n        block_size=62, share_weight=False, shared_codebook=False,\n        num_quantizers=6\n    )\n    with torch.no_grad():\n        if condition == "no_text":\n            y = proc.emb_cond(torch.randn(2, 12, 512))\n            assert torch.count_nonzero(y).item() == 0\n        elif condition == "no_pose":\n            y = proc.emb_pose_codes(torch.randn(2, 49, 394))\n            assert torch.count_nonzero(y).item() == 0\n        elif condition == "no_stage":\n            y = proc.emb_q_ids(torch.tensor([0, 3]))\n            assert torch.count_nonzero(y).item() == 0\n        elif condition == "no_history":\n            idx = torch.zeros(2, 49, 1, dtype=torch.long)\n            y = proc.emb_res_codes(idx, torch.ones(2, dtype=torch.long), inference=True)\n            assert torch.count_nonzero(y).item() == 0\n        else:\n            y = proc.emb_cond(torch.randn(2, 12, 512))\n            assert y.shape == (2, 12, 1024)\n    print(f"[OK] Dry-run contract passed: {condition}")\n    raise SystemExit(0)\n\nsrc_path = PGR / "eval_refine_trans.py"\nsrc = src_path.read_text()\n\nsrc = src.replace(\n    \'os.environ["CUDA_VISIBLE_DEVICES"] = "1"\',\n    \'# EXP04C: CUDA visibility is controlled by the wrapper\'\n)\n\nneedle = "repeat_time = 20"\nif src.count(needle) != 1:\n    raise RuntimeError(f"Expected exactly one {needle!r}, found {src.count(needle)}")\nsrc = src.replace(\n    needle,\n    \'repeat_time = int(os.environ.get("EXP04C_REPEATS", "20"))\'\n)\n\ncode = compile(src, str(src_path), "exec")\nexec(code, {"__name__": "__main__", "__file__": str(src_path)})\n'
WRAPPER.write_text(wrapper_text)
compile(WRAPPER.read_text(), str(WRAPPER), 'exec')
print('[OK] Wrapper:', WRAPPER)


In [ ]:

# 3. Fast intervention-contract preflight for all five conditions.
# This does NOT load checkpoints or run the HumanML3D evaluator.

CONDITIONS = ["full", "no_text", "no_pose", "no_stage", "no_history"]

for cond in CONDITIONS:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["EXP04C_CONDITION"] = cond
    env["EXP04C_DRYRUN"] = "1"
    env["EXP04C_REPEATS"] = "1"
    p = subprocess.run(
        [str(PY), "-u", str(WRAPPER)],
        cwd=PGR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, timeout=180,
    )
    print(f"\n=== {cond} ===")
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"Intervention contract failed: {cond}")

print("\n[OK] All five Exp.04C intervention contracts passed.")


In [ ]:

# 4. Freeze the production evaluation protocol.

REPEATS = 20
SEED = 123
EVAL_MODE = "fast"

COMMON = [
    "--exp-name", "EXP04C_PLACEHOLDER",
    "--out-dir", str(EVAL_OUT),
    "--dataname", "t2m",
    "--dec-checkpoint-folder", str(TOKENIZER_DIR),
    "--t2m-checkpoint-folder", str(BASE_DIR),
    "--residual-t2m-checkpoint-folder", str(REFINE_DIR),
    "--eval-reference", "net_best_fid",
    "--eval-mode", EVAL_MODE,
    "--use-keywords",
    "--seed", str(SEED),
]

FREEZE = RESULTS / "exp04c_protocol_freeze.json"
freeze = {
    "experiment": "Exp04C fixed-checkpoint Refine conditioning intervention",
    "conditions": CONDITIONS,
    "repeats_per_condition": REPEATS,
    "seed_reset_per_condition": SEED,
    "common_random_numbers": True,
    "eval_mode": EVAL_MODE,
    "test_split": "HumanML3D",
    "refine_checkpoint": str(REFINE_CKPT),
    "refine_checkpoint_sha256": sha256(REFINE_CKPT),
    "refine_checkpoint_nb_iter": int(b["checkpoint_nb_iter"]),
    "base_checkpoint": str(BASE_CKPT),
    "base_checkpoint_sha256": sha256(BASE_CKPT),
    "tokenizer_checkpoint": str(TOKENIZER_CKPT),
    "tokenizer_checkpoint_sha256": sha256(TOKENIZER_CKPT),
    "interpretation": "fixed-checkpoint causal sensitivity, not matched retraining",
    "no_text_scope": "Refine-stage condition only; Base Transformer retains text/keywords",
    "no_stage_scope": "explicit q embedding only; q-specific output head/history indexing retained",
}
FREEZE.write_text(json.dumps(freeze, indent=2))
print("[OK] Frozen protocol:", FREEZE)
print(json.dumps(freeze, indent=2))


In [ ]:

# 5. One-repeat FULL evaluator smoke.
# Run this before launching the full queue.

SMOKE_LOG = LOGDIR / "exp04c_full_smoke.log"

smoke_args = [
    x if x != "EXP04C_PLACEHOLDER" else "exp04c_full_smoke"
    for x in COMMON
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["EXP04C_CONDITION"] = "full"
env["EXP04C_REPEATS"] = "1"
env["EXP04C_DRYRUN"] = "0"

cmd = [str(PY), "-u", str(WRAPPER), *smoke_args]
print("=== EXP.04C FULL 1-REPEAT SMOKE ===")
print(" ".join(shlex.quote(x) for x in cmd))

t0 = time.time()
with open(SMOKE_LOG, "w") as f:
    p = subprocess.run(
        cmd, cwd=PGR, env=env,
        stdout=f, stderr=subprocess.STDOUT,
        timeout=21600,
    )
elapsed = (time.time() - t0) / 60

txt = SMOKE_LOG.read_text(errors="replace")
print("returncode:", p.returncode)
print(f"elapsed: {elapsed:.1f} min")
print("\n".join(txt.splitlines()[-120:]))

if p.returncode != 0:
    raise RuntimeError(f"Exp.04C FULL evaluator smoke failed: {SMOKE_LOG}")

required = [
    "Exp.04C intervention installed: full",
    "loading residual transformer checkpoint",
    "final result:",
]
missing = [m for m in required if m.lower() not in txt.lower()]
if missing:
    raise RuntimeError(f"Smoke returned 0 but markers missing: {missing}")

print("[OK] Exp.04C FULL evaluator smoke passed.")


In [ ]:
# 6. Build the resumable five-condition evaluation queue.
queue_text = '\nimport os, subprocess, json\nfrom pathlib import Path\n\nPGR = Path(".")\nPY = Path("venv/bin/python")\nPRE = PGR / "revision_preflight"\nRESULTS = PGR / "revision_results" / "exp04c_conditioning"\nLOGDIR = PGR / "revision_logs" / "exp04c_conditioning"\nEVAL_OUT = RESULTS / "eval_runs"\nWRAPPER = PRE / "eval_refine_trans_exp04c_intervention.py"\n\nb = json.loads((PGR/"revision_results"/"exp04b_refine"/"exp04b_refine_completion_audit.json").read_text())\nREFINE_DIR = Path(b["run_dir"])\nBASE_DIR = Path(b["base_dir"])\nTOKENIZER_DIR = Path(b["tokenizer_dir"])\n\nconditions = ["full", "no_text", "no_pose", "no_stage", "no_history"]\nrepeats = 20\nseed = 123\n\ndef args_for(cond):\n    return [\n        "--exp-name", f"exp04c_{cond}",\n        "--out-dir", str(EVAL_OUT),\n        "--dataname", "t2m",\n        "--dec-checkpoint-folder", str(TOKENIZER_DIR),\n        "--t2m-checkpoint-folder", str(BASE_DIR),\n        "--residual-t2m-checkpoint-folder", str(REFINE_DIR),\n        "--eval-reference", "net_best_fid",\n        "--eval-mode", "fast",\n        "--use-keywords",\n        "--seed", str(seed),\n    ]\n\nfor cond in conditions:\n    log = LOGDIR / f"exp04c_{cond}.log"\n    old = log.read_text(errors="replace") if log.exists() else ""\n    if "final result:" in old and "conf." in old:\n        print(f"[SKIP] {cond}: completed", flush=True)\n        continue\n\n    env = os.environ.copy()\n    env["CUDA_VISIBLE_DEVICES"] = "0"\n    env["EXP04C_CONDITION"] = cond\n    env["EXP04C_REPEATS"] = str(repeats)\n    env["EXP04C_DRYRUN"] = "0"\n\n    cmd = [str(PY), "-u", str(WRAPPER), *args_for(cond)]\n    print(f"[RUN] {cond}", flush=True)\n    print(" ".join(cmd), flush=True)\n\n    with open(log, "w") as f:\n        p = subprocess.run(\n            cmd, cwd=PGR, env=env,\n            stdout=f, stderr=subprocess.STDOUT\n        )\n\n    print(f"[DONE] {cond}: returncode={p.returncode}", flush=True)\n    if p.returncode != 0:\n        tail = "\\n".join(log.read_text(errors="replace").splitlines()[-100:])\n        print(tail, flush=True)\n        raise SystemExit(p.returncode)\n\nprint("[OK] EXP04C_QUEUE_COMPLETE", flush=True)\n'
QUEUE.write_text(queue_text)
compile(QUEUE.read_text(), str(QUEUE), 'exec')
print('[OK] Queue:', QUEUE)


In [ ]:

# 7. Launch the 5-condition x 20-repeat queue in the background.
# Run ONLY after Cell 5 prints the [OK] smoke marker.

def pid_alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

old_pid = None
if PIDFILE.exists():
    try:
        old_pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

old_txt = QUEUE_LOG.read_text(errors="replace") if QUEUE_LOG.exists() else ""
if "[OK] EXP04C_QUEUE_COMPLETE" in old_txt:
    print("=== EXP.04C QUEUE ALREADY COMPLETE ===")
elif old_pid and pid_alive(old_pid):
    print("=== EXP.04C QUEUE ALREADY RUNNING ===")
    print("PID:", old_pid)
else:
    handle = open(QUEUE_LOG, "a")
    p = subprocess.Popen(
        [str(PY), "-u", str(QUEUE)],
        cwd=PGR,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    PIDFILE.write_text(str(p.pid))
    LAUNCH_JSON.write_text(json.dumps({
        "pid": p.pid,
        "queue": str(QUEUE),
        "queue_log": str(QUEUE_LOG),
        "conditions": CONDITIONS,
        "repeats": REPEATS,
        "launched_at": time.time(),
    }, indent=2))
    time.sleep(1)

    print("=== EXP.04C QUEUE LAUNCH ===")
    print("PID   :", p.pid)
    print("Alive :", pid_alive(p.pid))
    print("Log   :", QUEUE_LOG)


In [ ]:

# 8. Read-only Exp.04C queue status. Safe to rerun.

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def count_runs(path):
    if not path.exists():
        return 0
    n = 0
    for line in path.read_text(errors="replace").splitlines():
        # eval_trans logs each repeat twice; count only the second line.
        if (
            re.search(r"^\d{4}-\d{2}-\d{2} .*Eva\. Iter 0\s*:, FID\.", line)
            and "multimodality. (N/A)" in line
        ):
            n += 1
    return n

pid = None
if PIDFILE.exists():
    try:
        pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

print("=== EXP.04C CONDITIONING STATUS ===")
print("PID   :", pid)
print("Alive :", pid_alive(pid) if pid else False)

total_done = 0
for cond in CONDITIONS:
    log = LOGDIR / f"exp04c_{cond}.log"
    n = count_runs(log)
    total_done += min(n, REPEATS)
    complete = False
    if log.exists():
        t = log.read_text(errors="replace")
        complete = "final result:" in t and "conf." in t
    print(f"{cond:10s}: {n:2d}/{REPEATS}  complete={complete}")

den = len(CONDITIONS) * REPEATS
print(f"Overall: {total_done}/{den} = {100*total_done/den:.1f}%")

print("\n=== QUEUE LOG TAIL ===")
if QUEUE_LOG.exists():
    print("\n".join(QUEUE_LOG.read_text(errors="replace").splitlines()[-60:]))


In [ ]:

# Standalone initialization for the final analysis cell.
from pathlib import Path
import json, re, hashlib
import numpy as np
import pandas as pd

PGR = Path(".")
RESULTS = PGR / "revision_results" / "exp04c_conditioning"
LOGDIR = PGR / "revision_logs" / "exp04c_conditioning"

CONDITIONS = ["full", "no_text", "no_pose", "no_stage", "no_history"]
REPEATS = 20
SEED = 123
EVAL_MODE = "fast"

B_AUDIT = PGR / "revision_results" / "exp04b_refine" / "exp04b_refine_completion_audit.json"
if not B_AUDIT.exists():
    raise RuntimeError(f"Missing Exp.04B audit: {B_AUDIT}")
b = json.loads(B_AUDIT.read_text())

REFINE_CKPT = Path(b["best_fid_checkpoint"])
if not REFINE_CKPT.exists():
    raise RuntimeError(f"Missing Refine checkpoint: {REFINE_CKPT}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()



# 9. Parse repeated metrics, paired degradations, and save reviewer-ready tables.
# Run after all five conditions are complete.

import numpy as np
import pandas as pd

RUN_RE = re.compile(
    r"FID\.\s*([0-9.eE+-]+),\s*"
    r"Diversity Real\.\s*([0-9.eE+-]+),\s*"
    r"Diversity\.\s*([0-9.eE+-]+),\s*"
    r"R_precision_real\.\s*\[([^\]]+)\],\s*"
    r"R_precision\.\s*\[([^\]]+)\],\s*"
    r"matching_score_real\.\s*([0-9.eE+-]+),\s*"
    r"matching_score_pred\.\s*([0-9.eE+-]+)"
)

def parse_vec(s):
    return [float(x) for x in s.replace(",", " ").split()]

rows = []
for cond in CONDITIONS:
    log = LOGDIR / f"exp04c_{cond}.log"
    if not log.exists():
        raise RuntimeError(f"Missing log: {log}")

    parsed = []
    for line in log.read_text(errors="replace").splitlines():
        # eval_trans writes two timestamped metric lines per repeat.
        # Keep only the second one, marked with multimodality. (N/A).
        if not (
            re.search(r"^\d{4}-\d{2}-\d{2} .*Eva\. Iter 0\s*:, FID\.", line)
            and "multimodality. (N/A)" in line
        ):
            continue
        m = RUN_RE.search(line)
        if not m:
            continue
        rp_real = parse_vec(m.group(4))
        rp = parse_vec(m.group(5))
        parsed.append({
            "fid": float(m.group(1)),
            "diversity_real": float(m.group(2)),
            "diversity": float(m.group(3)),
            "top1_real": rp_real[0],
            "top2_real": rp_real[1],
            "top3_real": rp_real[2],
            "top1": rp[0],
            "top2": rp[1],
            "top3": rp[2],
            "matching_real": float(m.group(6)),
            "matching": float(m.group(7)),
        })

    if len(parsed) != REPEATS:
        raise RuntimeError(
            f"{cond}: expected {REPEATS} timestamped evaluation rows, got {len(parsed)}"
        )

    for i, r in enumerate(parsed):
        rows.append({"condition": cond, "repeat": i, **r})

df = pd.DataFrame(rows)
repeat_csv = RESULTS / "exp04c_repeat_metrics.csv"
df.to_csv(repeat_csv, index=False)

metrics = ["fid", "diversity", "top1", "top2", "top3", "matching"]
summary_rows = []
for cond in CONDITIONS:
    d = df[df.condition == cond]
    row = {"condition": cond}
    for m in metrics:
        x = d[m].to_numpy()
        row[f"{m}_mean"] = float(x.mean())
        row[f"{m}_ci95"] = float(x.std(ddof=0) * 1.96 / np.sqrt(len(x)))
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary_csv = RESULTS / "exp04c_summary.csv"
summary.to_csv(summary_csv, index=False)

full = df[df.condition == "full"].sort_values("repeat").reset_index(drop=True)
paired_rows = []
for cond in CONDITIONS[1:]:
    d = df[df.condition == cond].sort_values("repeat").reset_index(drop=True)
    assert np.array_equal(full["repeat"].to_numpy(), d["repeat"].to_numpy())

    diffs = {
        "fid_increase": d["fid"].to_numpy() - full["fid"].to_numpy(),
        "matching_increase": d["matching"].to_numpy() - full["matching"].to_numpy(),
        "top1_drop": full["top1"].to_numpy() - d["top1"].to_numpy(),
        "top2_drop": full["top2"].to_numpy() - d["top2"].to_numpy(),
        "top3_drop": full["top3"].to_numpy() - d["top3"].to_numpy(),
    }

    row = {"condition": cond}
    for name, x in diffs.items():
        row[f"{name}_mean"] = float(x.mean())
        row[f"{name}_ci95"] = float(x.std(ddof=0) * 1.96 / np.sqrt(len(x)))
    paired_rows.append(row)

paired = pd.DataFrame(paired_rows)
paired_csv = RESULTS / "exp04c_paired_degradation.csv"
paired.to_csv(paired_csv, index=False)

audit = {
    "conditions": CONDITIONS,
    "repeats": REPEATS,
    "seed": SEED,
    "eval_mode": EVAL_MODE,
    "common_random_numbers": True,
    "refine_checkpoint": str(REFINE_CKPT),
    "refine_checkpoint_sha256": sha256(REFINE_CKPT),
    "repeat_metrics_csv": str(repeat_csv),
    "summary_csv": str(summary_csv),
    "paired_degradation_csv": str(paired_csv),
}
AUDIT = RESULTS / "exp04c_completion_audit.json"
AUDIT.write_text(json.dumps(audit, indent=2))

pd.set_option("display.max_columns", 50)
print("=== EXP.04C SUMMARY ===")
display(summary.round(6))
print("\n=== PAIRED DEGRADATION VS FULL ===")
display(paired.round(6))

print("\nSaved:")
print(" ", repeat_csv)
print(" ", summary_csv)
print(" ", paired_csv)
print(" ", AUDIT)
print("\n[OK] Exp.04C conditioning intervention analysis complete.")



## Interpretation guide

For the paired-degradation table, positive values mean the ablation hurt performance:

- `fid_increase > 0`: worse FID after removing the condition
- `matching_increase > 0`: worse matching score
- `top1_drop > 0`: lower Top-1 R-Precision
- `top2_drop > 0`: lower Top-2 R-Precision
- `top3_drop > 0`: lower Top-3 R-Precision

The strongest reviewer-facing evidence is a consistent paired degradation with a tight
confidence interval.

Recommended wording should remain narrow:

> “Fixed-checkpoint interventions indicate that the learned refinement model relies most
> strongly on ___, while removing ___ produces a smaller change.”

Do not describe these results as matched architectural retraining ablations.
